[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsasaki0109/slam-handbook-python/blob/main/part3_spatial_ai/ch15_dynamic_slam/01_dynamic_object_handling.ipynb)

# 第15章: 動的環境におけるSLAM

本ノートブックでは、動的物体がSLAMに与える影響と、その対処法を学びます。

## 学習内容
1. **動的物体の影響**: 動く物体がSLAMの推定精度をどう悪化させるか
2. **残差ベースの動的点検出**: 再投影誤差を用いた外れ値（動的点）の検出
3. **Before/After比較**: 動的点除去の効果の可視化

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
np.random.seed(42)

## 1. 動的シーンのシミュレーション

実環境では、歩行者や車両などの動的物体が存在します。SLAMは「世界は静的」という仮定のもとで動作するため、動的物体の特徴点は推定を大きく狂わせます。

以下では、静的な建物と動く歩行者を含む2Dシーンをシミュレートします。

In [ ]:
# --- Simulate a 2D scene with static and dynamic points ---

def create_scene():
    """Create a 2D scene with static landmarks and dynamic objects."""
    # Static landmarks (buildings, trees, etc.)
    static_points = np.array([
        [2, 8], [3, 9], [4, 7], [5, 10], [7, 8], [8, 9], [9, 7],
        [1, 5], [3, 4], [6, 5], [8, 3], [10, 6], [11, 4],
        [2, 1], [5, 2], [7, 1], [9, 2], [11, 1],
    ], dtype=float)
    
    # Dynamic points (pedestrians) - will move between frames
    dynamic_points_t0 = np.array([
        [4, 5], [4.5, 5.2], [5, 4.8],  # Person 1
        [8, 6], [8.3, 6.1], [7.8, 5.8],  # Person 2
    ], dtype=float)
    
    return static_points, dynamic_points_t0

def simulate_camera_motion(points, tx, ty, theta, noise_std=0.05):
    """Apply rigid body transformation + observation noise."""
    R = np.array([[np.cos(theta), -np.sin(theta)],
                  [np.sin(theta),  np.cos(theta)]])
    transformed = (R @ points.T).T + np.array([tx, ty])
    observed = transformed + np.random.randn(*transformed.shape) * noise_std
    return observed, R, np.array([tx, ty])

# Create scene
static_pts, dynamic_pts_t0 = create_scene()

# Camera moves: small translation + rotation
true_tx, true_ty, true_theta = 0.5, 0.3, 0.05

# Frame 1: observe all points
all_pts_t0 = np.vstack([static_pts, dynamic_pts_t0])
is_dynamic = np.array([False]*len(static_pts) + [True]*len(dynamic_pts_t0))

# Frame 2: static points transform with camera, dynamic points ALSO move independently
static_obs_t1, R_true, t_true = simulate_camera_motion(static_pts, true_tx, true_ty, true_theta)

# Dynamic objects move on their own (e.g., walking)
dynamic_pts_t1 = dynamic_pts_t0 + np.array([1.5, 0.8])  # pedestrians walked
dynamic_obs_t1, _, _ = simulate_camera_motion(dynamic_pts_t1, true_tx, true_ty, true_theta, noise_std=0.05)

all_obs_t1 = np.vstack([static_obs_t1, dynamic_obs_t1])

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, pts, title in [(axes[0], all_pts_t0, 'Frame 0 (Original)'),
                        (axes[1], all_obs_t1, 'Frame 1 (After Camera + Object Motion)')]:
    static_mask = ~is_dynamic
    ax.scatter(pts[static_mask, 0], pts[static_mask, 1], c='blue', s=80, marker='s', 
               label='Static landmarks', zorder=5)
    ax.scatter(pts[~static_mask, 0], pts[~static_mask, 1], c='red', s=80, marker='o',
               label='Dynamic objects', zorder=5)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-1, 13)
    ax.set_ylim(-1, 12)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()
print(f"静的点: {len(static_pts)}個, 動的点: {len(dynamic_pts_t0)}個")

## 2. 動的点がポーズ推定に与える影響

動的物体の特徴点を含めてポーズ推定（最小二乗法）を行うと、推定結果がバイアスされます。ここでは、全点を使った推定と静的点のみの推定を比較します。

In [ ]:
# --- Pose estimation with and without dynamic points ---

def estimate_rigid_transform_2d(src, dst):
    """Estimate 2D rigid transform (R, t) from point correspondences using SVD."""
    centroid_src = np.mean(src, axis=0)
    centroid_dst = np.mean(dst, axis=0)
    src_centered = src - centroid_src
    dst_centered = dst - centroid_dst
    H = src_centered.T @ dst_centered
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T
    t = centroid_dst - R @ centroid_src
    return R, t

def compute_residuals(src, dst, R, t):
    """Compute per-point residual after applying estimated transform."""
    transformed = (R @ src.T).T + t
    residuals = np.linalg.norm(transformed - dst, axis=1)
    return residuals

# Estimate with ALL points (including dynamic)
R_all, t_all = estimate_rigid_transform_2d(all_pts_t0, all_obs_t1)

# Estimate with STATIC points only (oracle)
R_static, t_static = estimate_rigid_transform_2d(static_pts, static_obs_t1)

# Compute errors
theta_all = np.arctan2(R_all[1, 0], R_all[0, 0])
theta_static = np.arctan2(R_static[1, 0], R_static[0, 0])

print("=== Pose Estimation Results ===")
print(f"\nGround truth: tx={true_tx:.3f}, ty={true_ty:.3f}, theta={true_theta:.4f} rad")
print(f"\nWith ALL points (dynamic included):")
print(f"  tx={t_all[0]:.3f}, ty={t_all[1]:.3f}, theta={theta_all:.4f} rad")
print(f"  Translation error: {np.linalg.norm(t_all - t_true):.4f}")
print(f"  Rotation error: {abs(theta_all - true_theta):.4f} rad")
print(f"\nWith STATIC points only:")
print(f"  tx={t_static[0]:.3f}, ty={t_static[1]:.3f}, theta={theta_static:.4f} rad")
print(f"  Translation error: {np.linalg.norm(t_static - t_true):.4f}")
print(f"  Rotation error: {abs(theta_static - true_theta):.4f} rad")

# Compute residuals for dynamic point detection
residuals = compute_residuals(all_pts_t0, all_obs_t1, R_static, t_static)
print(f"\n--- Residuals ---")
for i, r in enumerate(residuals):
    label = "DYNAMIC" if is_dynamic[i] else "static"
    print(f"  Point {i:2d} [{label:7s}]: residual = {r:.4f}")

## 3. 残差ベースの動的点検出と除去

反復的な手法で動的点を検出します：
1. 全点でポーズを推定
2. 各点の残差（再投影誤差）を計算
3. 閾値以上の残差を持つ点を動的と判定して除外
4. 残った点で再度ポーズを推定（反復）

In [ ]:
# --- Iterative dynamic point detection and removal ---

def iterative_dynamic_detection(src, dst, threshold=0.5, max_iter=5):
    """Detect dynamic points by iterative residual thresholding."""
    n = len(src)
    inlier_mask = np.ones(n, dtype=bool)
    history = []
    
    for iteration in range(max_iter):
        # Estimate transform using current inliers
        R, t = estimate_rigid_transform_2d(src[inlier_mask], dst[inlier_mask])
        
        # Compute residuals for ALL points
        residuals = compute_residuals(src, dst, R, t)
        
        # Update inlier mask
        new_mask = residuals < threshold
        history.append({
            'iteration': iteration,
            'n_inliers': np.sum(new_mask),
            'R': R, 't': t,
            'residuals': residuals.copy(),
            'mask': new_mask.copy()
        })
        
        if np.array_equal(new_mask, inlier_mask):
            break
        inlier_mask = new_mask
    
    return history

history = iterative_dynamic_detection(all_pts_t0, all_obs_t1, threshold=0.5, max_iter=5)

# Final result
final = history[-1]
detected_dynamic = ~final['mask']

# Visualization: Before vs After
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Before: all points used
transformed_all = (R_all @ all_pts_t0.T).T + t_all
axes[0].scatter(all_obs_t1[~is_dynamic, 0], all_obs_t1[~is_dynamic, 1], c='blue', s=60, 
                marker='s', label='Static (observed)', zorder=5)
axes[0].scatter(all_obs_t1[is_dynamic, 0], all_obs_t1[is_dynamic, 1], c='red', s=60,
                marker='o', label='Dynamic (observed)', zorder=5)
axes[0].scatter(transformed_all[:, 0], transformed_all[:, 1], c='gray', s=30, marker='x',
                label='Predicted (all pts)', alpha=0.7)
for i in range(len(all_pts_t0)):
    axes[0].plot([all_obs_t1[i, 0], transformed_all[i, 0]], 
                 [all_obs_t1[i, 1], transformed_all[i, 1]], 'k-', alpha=0.2)
axes[0].set_title('Before: Using All Points', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_aspect('equal')

# Residual plot
colors = ['red' if d else 'blue' for d in is_dynamic]
axes[1].bar(range(len(final['residuals'])), final['residuals'], color=colors, alpha=0.7)
axes[1].axhline(y=0.5, color='green', linestyle='--', linewidth=2, label='Threshold')
axes[1].set_xlabel('Point Index')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residuals (red=dynamic, blue=static)', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

# After: dynamic points removed
R_final, t_final = final['R'], final['t']
inlier_src = all_pts_t0[final['mask']]
inlier_dst = all_obs_t1[final['mask']]
transformed_final = (R_final @ inlier_src.T).T + t_final

axes[2].scatter(inlier_dst[:, 0], inlier_dst[:, 1], c='blue', s=60, marker='s',
                label='Inlier points', zorder=5)
axes[2].scatter(all_obs_t1[detected_dynamic, 0], all_obs_t1[detected_dynamic, 1], 
                c='red', s=60, marker='x', label='Rejected (dynamic)', zorder=5, alpha=0.5)
axes[2].scatter(transformed_final[:, 0], transformed_final[:, 1], c='green', s=30, marker='+',
                label='Predicted (inliers only)', zorder=4)
axes[2].set_title('After: Dynamic Points Removed', fontsize=13, fontweight='bold')
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)
axes[2].set_aspect('equal')

plt.tight_layout()
plt.savefig('dynamic_slam.png', dpi=100, bbox_inches='tight')
plt.show()

# Error comparison
theta_final = np.arctan2(R_final[1, 0], R_final[0, 0])
print(f"\n=== 改善結果 ===")
print(f"動的点使用時の並進誤差: {np.linalg.norm(t_all - t_true):.4f}")
print(f"動的点除去後の並進誤差: {np.linalg.norm(t_final - t_true):.4f}")
print(f"検出された動的点: {np.sum(detected_dynamic)}個 (真値: {np.sum(is_dynamic)}個)")
print(f"検出精度: {np.sum(detected_dynamic & is_dynamic)}/{np.sum(is_dynamic)} 正しく検出")

## 演習問題

1. **動的物体の割合**: 動的点の数を増やして（全体の50%以上にして）、検出アルゴリズムが破綻する条件を調べてください。
2. **閾値の影響**: `threshold` を 0.1, 0.3, 0.5, 1.0 と変えて、検出精度への影響を観察してください。
3. **RANSAC**: 残差ベースの反復法の代わりに、RANSACベースのロバスト推定を実装してみてください。

## まとめ

- **動的物体**はSLAMの「静的世界仮定」に反し、ポーズ推定に大きな誤差を引き起こす
- **残差ベースの検出**は、再投影誤差が大きい点を動的と判定するシンプルかつ効果的な手法
- 実際のシステムでは、**セマンティックセグメンテーション**（車、人などのクラス情報）と組み合わせることで、より確実な動的物体の除去が可能
- 動的物体自体の運動推定（**動的SLAM**）も研究の活発な分野であり、自動運転等で重要